In [ ]:
import os
import requests
import asyncio

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.runtime import SingleThreadedAgentRuntime
from autogen_agentchat.messages import Message
from autogen_agentchat.types import AgentId
from dotenv import load_dotenv


In [ ]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
github_token = os.getenv("GITHUB_TOKEN")

### GitHub PR Fetcher

In [ ]:
def get_pr_files(owner, repo, pr_number):
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}/files"

    headers = {
        "Authorization": f"Bearer {os.getenv('GITHUB_TOKEN')}"
    }

    response = requests.get(url, headers=headers)
    return response.json()

### Model Client

In [ ]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini"
)

### PR Reader Agent

In [ ]:
pr_reader = AssistantAgent(
    name="pr_reader",
    model_client=model_client,
    system_message="""
You analyze GitHub Pull Request code changes.
Explain what the PR is doing in simple terms.
"""
)

### Code Reviewer Agent

In [ ]:
code_reviewer = AssistantAgent(
    name="code_reviewer",
    model_client=model_client,
    system_message="""
You are a senior software engineer reviewing a pull request.

Find:
- bugs
- poor code structure
- missing validation
- performance issues
"""
)

### Security Agent

In [ ]:
security_agent = AssistantAgent(
    name="security_agent",
    model_client=model_client,
    system_message="""
You are a security engineer reviewing code.

Check for:
- exposed secrets
- unsafe authentication
- injection vulnerabilities
- insecure API usage
"""
)

### Summary Agent

In [ ]:
summary_agent = AssistantAgent(
    name="summary_agent",
    model_client=model_client,
    system_message="""
Combine all previous reviews and produce a final PR report.

Format:

PR Summary
Issues
Security Risks
Recommendations
Risk Score (Low / Medium / High)
"""
)

### Setup Run time

In [ ]:
runtime = SingleThreadedAgentRuntime()

runtime.register_agent(pr_reader)
runtime.register_agent(code_reviewer)
runtime.register_agent(security_agent)
runtime.register_agent(summary_agent)

runtime.start()

### Fetch PR

In [ ]:
owner = input("Repo Owner: ")
repo = input("Repo Name: ")
pr_number = int(input("PR Number: "))

files = get_pr_files(owner, repo, pr_number)

pr_code = ""

for f in files:
    pr_code += f"\nFile: {f['filename']}\n"
    pr_code += f.get("patch", "")

### Agent Work Flow

In [ ]:
response1 = await runtime.send_message(
    Message(pr_code),
    AgentId("pr_reader", "default")
)

print("PR Reader Analysis:\n", response1.content)

### Code Reviewer

In [ ]:
response2 = await runtime.send_message(
    Message(response1.content),
    AgentId("code_reviewer", "default")
)

print("\nCode Review:\n", response2.content)

### Security Review

In [ ]:
response3 = await runtime.send_message(
    Message(response2.content),
    AgentId("security_agent", "default")
)

print("\nSecurity Review:\n", response3.content)

### Final. Summary

In [ ]:
response4 = await runtime.send_message(
    Message(response3.content),
    AgentId("summary_agent", "default")
)

print("\nFinal PR Report:\n")
print(response4.content)